# KV Cache Compression Methods — Comparison on LongBench v1

**Model**: meta-llama/Meta-Llama-3.1-8B-Instruct  
**Dataset**: LongBench v1 (16 tasks, 200 samples each)  
**Methods compared**:
| Label | Description |
|-------|-------------|
| **FullKV** | FP16 full KV cache baseline |
| **LiveKVQuantP** | Chunk-wise online INT4 quantization + outlier isolation (Ours) |
| **KVQuant** | Offline INT4 static KV quantization with CUDA kernels |
| **Finch** | Attention-score based token dropping (GQA-aware per-head vote, split_size=512) |

**Sections**:
1. Accuracy (Score)
2. End-to-End Latency
3. Peak Memory
4. Prefill / Decode Latency Breakdown — LiveKVQuantP vs KVQuant vs Finch


In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

TASKS = [
    'narrativeqa', 'qasper', 'multifieldqa_en', 'hotpotqa', '2wikimqa', 'musique',
    'gov_report', 'qmsum', 'multi_news', 'trec', 'triviaqa', 'samsum',
    'passage_retrieval_en', 'passage_count', 'lcc', 'repobench-p',
]
TASK_LABELS = [t.replace('_', '\n') for t in TASKS]

# ── load FullKV ──────────────────────────────────────────────────────────────
fullkv = {}
for f in glob.glob('../results/fullKV/longbench_v1_results/*.json'):
    d = json.load(open(f))
    fullkv[d['task']] = {
        'score': d['avg_score'],
        'e2e':   d['avg_latency_ms'],
        'mem':   d['max_peak_memory_mb'],
    }

# ── load LiveKVQuantP ────────────────────────────────────────────────────────
lkp = {}
for f in glob.glob('../results/liveKVQuant/longbench_v1_result/*.json'):
    d = json.load(open(f))
    lkp[d['task']] = {
        'score':   d['avg_score'],
        'e2e':     d['avg_end_to_end_latency_ms'],
        'prefill': d['avg_prefill_latency_ms'],
        'decode':  d['avg_decode_latency_ms'],
        'mem':     d['max_peak_memory_mb'],
    }

# ── load KVQuant ─────────────────────────────────────────────────────────────
kvq = {}
for f in glob.glob('../../KVQuant/results/KVQuant/longbench_v1_results/*.json'):
    d = json.load(open(f))
    r = d['results']
    kvq[d['task']] = {
        'score':   r['avg_score'],
        'e2e':     r['avg_end_to_end_latency_ms'],
        'prefill': r['avg_prefill_latency_ms'],
        'decode':  r['avg_decode_latency_ms'],
        'mem':     r['max_peak_memory_mb'],
    }

# ── load Finch v3 ─────────────────────────────────────────────────────────────
finch = {}
v3_dir = '../../Finch/logs/longbench_v1_results/v3_result (new prompt template and penalty=1.0)/'
for f in glob.glob(v3_dir + '*.json'):
    d = json.load(open(f))
    task = d['configs']['task_name']
    r    = d['result']
    raw_score = r['avg_f1_score']
    finch[task] = {
        'score':   raw_score / 100.0,   # Finch stores as 0–100
        'e2e':     r['avg_e2e_latency_ms'],
        'prefill': r['avg_prefill_latency_ms'],
        'decode':  r['avg_decode_latency_ms'],
        'mem':     r['max_peak_memory'],
    }

print(f"Tasks loaded — FullKV: {len(fullkv)}, LKP: {len(lkp)}, KVQ: {len(kvq)}, Finch: {len(finch)}")
assert all(len(x) == 16 for x in [fullkv, lkp, kvq, finch]), "Missing tasks!"
print("All 16 tasks loaded ✓")


In [ ]:
def build_df(metric, sources, labels):
    """Build a tasks × methods DataFrame for a given metric key."""
    rows = {}
    for task in TASKS:
        row = {}
        for src, lbl in zip(sources, labels):
            v = src.get(task, {}).get(metric)
            row[lbl] = v
        rows[task] = row
    df = pd.DataFrame(rows).T  # tasks as rows
    df.index.name = 'Task'
    return df[labels]   # column order

METHODS  = ['FullKV', 'LiveKVQuantP', 'KVQuant', 'Finch']
SOURCES  = [fullkv, lkp, kvq, finch]
COLORS   = ['#4C72B0', '#55A868', '#C44E52', '#DD8452']
METHODS3 = ['LiveKVQuantP', 'KVQuant', 'Finch']
COLORS3  = ['#55A868', '#C44E52', '#DD8452']

df_score   = build_df('score', SOURCES, METHODS)
df_e2e     = build_df('e2e',   SOURCES, METHODS)
df_mem     = build_df('mem',   SOURCES, METHODS)
df_prefill = build_df('prefill', [lkp, kvq, finch], METHODS3)
df_decode  = build_df('decode',  [lkp, kvq, finch], METHODS3)

# Average row (only numeric, skip NaN for FullKV e2e if any)
for df in [df_score, df_e2e, df_mem, df_prefill, df_decode]:
    df.loc['Average'] = df.mean(skipna=True)

print("DataFrames built ✓")


---
## Section 1 — Accuracy (Score)

Each task scored with its official LongBench v1 metric (F1 / ROUGE-L / Accuracy / Edit-sim).  
Scores normalised to **0–1**.


In [ ]:
def fmt_delta(val, ref):
    d = val - ref
    s = f'{d:+.4f}'
    return s

display_df = df_score.copy()
display_df['Δ LKP'] = df_score['LiveKVQuantP'] - df_score['FullKV']
display_df['Δ KVQ'] = df_score['KVQuant']      - df_score['FullKV']
display_df['Δ Finch']= df_score['Finch']        - df_score['FullKV']

styled = (display_df.style
    .format({
        'FullKV': '{:.4f}', 'LiveKVQuantP': '{:.4f}',
        'KVQuant': '{:.4f}', 'Finch': '{:.4f}',
        'Δ LKP': '{:+.4f}', 'Δ KVQ': '{:+.4f}', 'Δ Finch': '{:+.4f}',
    })
    .background_gradient(subset=['Δ LKP','Δ KVQ','Δ Finch'],
                         cmap='RdYlGn', vmin=-0.15, vmax=0.02)
    .set_caption('Table 1 — Score (0–1). Δ = method − FullKV.')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size','13px'),('font-weight','bold'),
                                  ('text-align','left'),('margin-bottom','6px')]}])
)
styled


In [ ]:
fig, ax = plt.subplots(figsize=(16, 4.5))
x     = np.arange(len(TASKS))
width = 0.20

for i, (method, color) in enumerate(zip(METHODS, COLORS)):
    vals = [df_score.loc[t, method] for t in TASKS]
    bars = ax.bar(x + (i - 1.5) * width, vals, width,
                  label=method, color=color, alpha=0.88, edgecolor='white', linewidth=0.4)

ax.set_xticks(x)
ax.set_xticklabels(TASKS, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Score (0–1)')
ax.set_title('Figure 1 — Per-task Score Comparison')
ax.legend(framealpha=0.3, fontsize=9)
ax.set_ylim(0, 1.12)
ax.axhline(0, color='k', linewidth=0.4)
plt.tight_layout()
plt.savefig('figures/score_comparison.png', bbox_inches='tight')
plt.show()


### Why does Finch score significantly lower on certain tasks?

Finch performs **attention-score based token dropping** — it permanently discards KV cache entries
judged unimportant. Several structural mismatches with Llama 3.1 8B make this selection unreliable:

| Root cause | Detail |
|-----------|--------|
| **GQA weakens voting signal** | Llama 3.1 uses GQA with only **8 KV heads** (vs. 32 in MHA). Token importance is averaged across fewer independent attention heads → noisier selection. |
| **Flat attention from large RoPE θ** | Llama 3.1 sets θ = **500 k** (vs. 10 k in Llama 2). This flattens attention distributions over long distances, making it hard to distinguish important from unimportant tokens. |
| **Iterative compression error** | Context is processed in split-size chunks (512 tokens). Each chunk discards tokens that earlier chunks may have needed → errors accumulate toward the end of long documents. |

Tasks most affected: **passage_retrieval_en** (−0.83), **triviaqa** (−0.08), **hotpotqa** (−0.27) —
all require locating a specific span or fact scattered across a long context.

---

### Why does KVQuant score slightly lower on passage_count?

KVQuant pre-allocates a **static KV buffer of `maxseqlen = 32 768` tokens**.  
For tasks whose samples exceed that limit, the input is silently **truncated**, causing
those samples to fail. `passage_count` has inputs that can exceed 32 k tokens, meaning
several samples produce wrong answers regardless of quantisation quality.  
*(See `KVQuant/docs/研究過程.md` — Bug 8: maxseqlen truncation.)*


---
## Section 2 — End-to-End Latency

E2E latency = total wall-clock time per sample (prefill + decode).  
Unit: **ms**. Lower is better.


In [ ]:
display_e2e = df_e2e.copy()
for m in ['LiveKVQuantP', 'KVQuant', 'Finch']:
    display_e2e[f'{m} overhead'] = df_e2e[m] / df_e2e['FullKV']

styled_e2e = (display_e2e.style
    .format({
        'FullKV': '{:,.0f}', 'LiveKVQuantP': '{:,.0f}',
        'KVQuant': '{:,.0f}', 'Finch': '{:,.0f}',
        'LiveKVQuantP overhead': '{:.2f}x',
        'KVQuant overhead':      '{:.2f}x',
        'Finch overhead':        '{:.2f}x',
    })
    .background_gradient(subset=['LiveKVQuantP overhead','KVQuant overhead','Finch overhead'],
                         cmap='RdYlGn_r', vmin=0.8, vmax=4.0)
    .set_caption('Table 2 — E2E Latency (ms). Overhead = method / FullKV (>1.0x = slower).')
    .set_table_styles([{'selector':'caption',
                        'props':[('font-size','13px'),('font-weight','bold'),
                                 ('text-align','left'),('margin-bottom','6px')]}])
)
styled_e2e


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 4.5))

# Left: absolute latency
x = np.arange(len(TASKS))
width = 0.20
for i, (method, color) in enumerate(zip(METHODS, COLORS)):
    vals = [df_e2e.loc[t, method] / 1000 for t in TASKS]  # → seconds
    ax1.bar(x + (i-1.5)*width, vals, width, label=method, color=color, alpha=0.88,
            edgecolor='white', linewidth=0.4)
ax1.set_xticks(x); ax1.set_xticklabels(TASKS, rotation=35, ha='right', fontsize=9)
ax1.set_ylabel('E2E Latency (s)'); ax1.set_title('Figure 2a — E2E Latency (absolute)')
ax1.legend(framealpha=0.3, fontsize=9)

# Right: overhead vs FullKV (3 compressed methods only)
for i, (method, color) in enumerate(zip(METHODS3, COLORS3)):
    overheads = [df_e2e.loc[t, method] / df_e2e.loc[t, 'FullKV'] for t in TASKS]
    ax2.bar(x + (i-1)*width, overheads, width, label=method, color=color, alpha=0.88,
            edgecolor='white', linewidth=0.4)
ax2.axhline(1.0, color='k', linewidth=1, linestyle='--', alpha=0.5, label='FullKV baseline')
ax2.set_xticks(x); ax2.set_xticklabels(TASKS, rotation=35, ha='right', fontsize=9)
ax2.set_ylabel('Overhead vs FullKV'); ax2.set_title('Figure 2b — E2E Overhead (ratio vs FullKV)')
ax2.legend(framealpha=0.3, fontsize=9)

plt.tight_layout()
plt.savefig('figures/e2e_latency.png', bbox_inches='tight')
plt.show()


---
## Section 3 — Peak GPU Memory

Peak GPU memory (MB) measured during inference.  
Lower is better.


In [ ]:
display_mem = df_mem.copy()
for m in ['LiveKVQuantP', 'KVQuant', 'Finch']:
    display_mem[f'Δ {m} (MB)'] = df_mem[m] - df_mem['FullKV']

styled_mem = (display_mem.style
    .format({
        'FullKV': '{:,.0f}', 'LiveKVQuantP': '{:,.0f}',
        'KVQuant': '{:,.0f}', 'Finch': '{:,.0f}',
        'Δ LiveKVQuantP (MB)': '{:+,.0f}',
        'Δ KVQuant (MB)':      '{:+,.0f}',
        'Δ Finch (MB)':        '{:+,.0f}',
    })
    .background_gradient(subset=['Δ LiveKVQuantP (MB)','Δ KVQuant (MB)','Δ Finch (MB)'],
                         cmap='RdYlGn', vmin=-10000, vmax=3000)
    .set_caption('Table 3 — Peak GPU Memory (MB). Δ = method − FullKV (negative = less memory).')
    .set_table_styles([{'selector':'caption',
                        'props':[('font-size','13px'),('font-weight','bold'),
                                 ('text-align','left'),('margin-bottom','6px')]}])
)
styled_mem


In [ ]:
fig, ax = plt.subplots(figsize=(16, 4.5))
x = np.arange(len(TASKS))
width = 0.20
for i, (method, color) in enumerate(zip(METHODS, COLORS)):
    vals = [df_mem.loc[t, method] / 1024 for t in TASKS]  # → GB
    ax.bar(x + (i-1.5)*width, vals, width, label=method, color=color, alpha=0.88,
           edgecolor='white', linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(TASKS, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Peak Memory (GB)'); ax.set_title('Figure 3 — Peak GPU Memory Comparison')
ax.legend(framealpha=0.3, fontsize=9)
plt.tight_layout()
plt.savefig('figures/memory.png', bbox_inches='tight')
plt.show()


### Why does KVQuant not save much memory on short tasks?

KVQuant pre-allocates a **static INT4 KV buffer of `maxseqlen = 32 768` tokens** at model load time,
regardless of the actual input length.  
Even for a 2 k-token input (e.g. `qasper`), the full 32 k INT4 buffer is resident on the GPU,
leaving peak memory close to FullKV.

In contrast:
- **LiveKVQuantP** allocates KV buffers chunk-by-chunk; only the actual sequence length is stored → memory scales with input.
- **Finch** drops tokens (reducing KV cache size), but stores the remaining tokens in **FP16**, giving moderate but consistent memory savings.


---
## Section 4 — Prefill / Decode Latency Breakdown

Comparing **LiveKVQuantP**, **KVQuant**, and **Finch** only  
(FullKV does not have prefill/decode split instrumentation).

> **Prefill** = processing the full input prompt (KV construction)  
> **Decode** = autoregressive token generation


In [ ]:
# Build detailed breakdown DataFrame
rows = {}
for task in TASKS:
    row = {}
    for src, m in zip([lkp, kvq, finch], METHODS3):
        e2e     = src[task]['e2e']
        prefill = src[task]['prefill']
        decode  = src[task]['decode']
        row[f'{m} Prefill'] = prefill
        row[f'{m} Decode']  = decode
        row[f'{m} E2E']     = e2e
        row[f'{m} Prefill%']= prefill / e2e * 100
    rows[task] = row

df_breakdown = pd.DataFrame(rows).T
df_breakdown.index.name = 'Task'

# Average row
df_breakdown.loc['Average'] = df_breakdown.mean()

# Display summary: Prefill / Decode / Prefill% for each method
cols_show = []
for m in METHODS3:
    cols_show += [f'{m} Prefill', f'{m} Decode', f'{m} Prefill%']

fmt_dict = {}
for m in METHODS3:
    fmt_dict[f'{m} Prefill']  = '{:,.0f}'
    fmt_dict[f'{m} Decode']   = '{:,.0f}'
    fmt_dict[f'{m} Prefill%'] = '{:.1f}%'

(df_breakdown[cols_show].style
    .format(fmt_dict)
    .background_gradient(subset=[f'{m} Prefill%' for m in METHODS3],
                         cmap='Oranges', vmin=20, vmax=100)
    .set_caption('Table 4 — Prefill / Decode Latency (ms) and Prefill% per method.')
    .set_table_styles([{'selector':'caption',
                        'props':[('font-size','13px'),('font-weight','bold'),
                                 ('text-align','left'),('margin-bottom','6px')]}])
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=False)
PREFILL_COLORS = {'LiveKVQuantP': '#2d7e4a', 'KVQuant': '#9b1c1c', 'Finch': '#a0522d'}
DECODE_COLORS  = {'LiveKVQuantP': '#a8d5b5', 'KVQuant': '#f5a0a0', 'Finch': '#f5c79e'}

x = np.arange(len(TASKS))
w = 0.55

for ax, m in zip(axes, METHODS3):
    prefills = [df_breakdown.loc[t, f'{m} Prefill'] / 1000 for t in TASKS]
    decodes  = [df_breakdown.loc[t, f'{m} Decode']  / 1000 for t in TASKS]
    bar1 = ax.bar(x, prefills, w, label='Prefill', color=PREFILL_COLORS[m], alpha=0.9)
    bar2 = ax.bar(x, decodes,  w, bottom=prefills, label='Decode',
                  color=DECODE_COLORS[m], alpha=0.9)
    ax.set_title(f'{m}', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(TASKS, rotation=40, ha='right', fontsize=8)
    ax.set_ylabel('Latency (s)')
    ax.legend(fontsize=9, framealpha=0.3)

fig.suptitle('Figure 4 — Prefill vs Decode Latency Breakdown (s)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('figures/prefill_decode_breakdown.png', bbox_inches='tight')
plt.show()


In [ ]:
# Average Prefill% per method
avg_prefill_pct = {m: df_breakdown.loc['Average', f'{m} Prefill%'] for m in METHODS3}
avg_e2e         = {m: df_breakdown.loc['Average', f'{m} E2E'] / 1000 for m in METHODS3}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Bar: average prefill%
bars = ax1.bar(METHODS3, [avg_prefill_pct[m] for m in METHODS3],
               color=COLORS3, alpha=0.88, edgecolor='white', width=0.5)
ax1.set_ylabel('Avg Prefill Latency (%)')
ax1.set_title('Figure 5a — Average Prefill % of E2E')
ax1.set_ylim(0, 105)
for bar, m in zip(bars, METHODS3):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{avg_prefill_pct[m]:.1f}%', ha='center', va='bottom', fontsize=10)

# Bar: average E2E
bars2 = ax2.bar(METHODS3, [avg_e2e[m] for m in METHODS3],
                color=COLORS3, alpha=0.88, edgecolor='white', width=0.5)
ax2.set_ylabel('Avg E2E Latency (s)')
ax2.set_title('Figure 5b — Average E2E Latency')
for bar, m in zip(bars2, METHODS3):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{avg_e2e[m]:.1f}s', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('figures/avg_latency_summary.png', bbox_inches='tight')
plt.show()


### Why is KVQuant's prefill latency so dominant?

KVQuant applies quantisation **inline during prefill**: for every chunk of the input,
it runs a FP16 → INT4 CUDA kernel to compress the K/V cache before continuing.  
This means:

1. **Every prefill step writes to a compressed buffer** using custom CUDA kernels (`quant_and_pack_vcache` / `quant_and_pack_kcache`).
2. The kernels were originally written for **MHA** (1 KV head per attention head); supporting GQA required a `num_groups` patch, adding branching overhead.
3. The **RoPE scaling fix** for Llama 3.1 (per-frequency scaling inside the K-cache kernel) adds an extra computation pass during prefill.

Net effect: KVQuant's prefill is **~10× longer** than LiveKVQuantP on narrativeqa despite both operating on the same sequence length.  
*(See `KVQuant/docs/研究過程.md` §2026-03-19-140000 and `docs/重點.md`)*

---

### Why is Finch's prefill also slow?

Finch does **not** reduce the prefill computation — it must still attend over the full context
to compute attention scores for token selection.  
In fact, prefill is *slightly heavier* than FullKV because:
- Each split-size chunk (512 tokens) triggers an extra attention pass to score tokens.
- The score aggregation across GQA heads adds a small but non-zero per-chunk overhead.

Finch saves memory (by discarding tokens from the KV cache), but the saving only materialises
**after** the prefill is complete, so decode benefits while prefill does not.

---

### Why does LiveKVQuantP have lower E2E overhead than KVQuant?

LiveKVQuantP's chunk-wise design keeps quantisation in a **streaming pipeline**:
- Quantisation happens per chunk and immediately frees the FP16 activations.
- Uses SDPA (Flash-Attention) for the attention computation, avoiding the explicit O(n²) attention matrix.
- INT4 storage is allocated dynamically, scaling with actual sequence length.

*(See `LiveKVQuantP/docs/memory_optimization_analysis.md` — Proposal A: SDPA fix, ~8 GB saved.)*


## Summary

| | FullKV | LiveKVQuantP | KVQuant | Finch |
|---|---|---|---|---|
| **Avg Score** | 0.4981 | **0.4860** (−0.0121) | 0.4929 (−0.0052) | 0.3189 (−0.1792) |
| **Avg E2E Latency** | 4.82s | 7.54s (+1.56×) | 10.78s (+2.24×) | 7.23s (+1.50×) |
| **Avg Prefill %** | — | 62.8% | 60.9% | 64.3% |
| **Avg Peak Memory** | 20.99 GB | 19.48 GB (−1.51) | 24.19 GB (+3.20) | 17.59 GB (−3.40) |
| **Llama 3.1 issues** | — | Minor | GQA kernel + static buf | GQA voting + flat attn |

**Key takeaways**:
- **LiveKVQuantP** has the best accuracy–efficiency balance: only −1.2% score with moderate latency overhead.
- **KVQuant** preserves accuracy well (−0.5%) but pays a large prefill cost (+2.24× E2E) due to inline FP16→INT4 kernels, and *uses more memory than FullKV* because of static `maxseqlen` pre-allocation.
- **Finch** gives the best memory savings (−3.4 GB avg) but accuracy drops sharply (−17.9%) due to GQA voting noise and flat attention distributions under Llama 3.1's large RoPE θ.

In [ ]:
print('=== Average Scores ===')
for m in METHODS:
    avg = df_score.loc[TASKS, m].mean()
    print(f'  {m:16s}: {avg:.4f}')

print()
print('=== Average E2E Latency (s) ===')
for m in METHODS:
    avg = df_e2e.loc[TASKS, m].mean() / 1000
    print(f'  {m:16s}: {avg:.2f}s')

print()
print('=== Average Prefill % ===')
for m in METHODS3:
    avg = df_breakdown.loc[TASKS, f'{m} Prefill%'].mean()
    print(f'  {m:16s}: {avg:.1f}%')

print()
print('=== Average Memory (GB) ===')
for m in METHODS:
    avg = df_mem.loc[TASKS, m].mean() / 1024
    print(f'  {m:16s}: {avg:.2f} GB')
